# Dữ liệu cần có #

#### Là dữ liệu hành vi  màn hình thi của thí sinh trong quá trình kiểm tra online ####

# Phương pháp thu thập dữ liệu #

- #### Thu thập dữ liệu thông qua việc chạy bài kiểm tra mô phỏng bao gồm 20 câu hỏi triết học. ####
- #### Dữ liệu bao gồm thời gian làm bài kiểm tra, số lần copy/paste, số lần chuyển cửa sổ,... ####
- ####  Mỗi lần chạy mô phỏng dữ liệu sẽ được lưu vào file csv với 1 test ID riêng việt. ####
- #### Trong quá trình thu thập dữ liệu đã có ít nhất 10 ứng viên được mời để thử nghiệm bài test và lấy mẫu dữ liệu và có hơn 400 mẫu dữ liệu đã được tạo ra ####

# Hợp nhất dữ liệu #

#### Tất cả các file bắt đầu bằng "test_results" sẽ được đọc và ghi vào file merged_test_results.csv ####

In [2]:
import os
import glob
import pandas as pd

folder_path = '.' 

file_pattern = os.path.join(folder_path, "test_results*.csv")
all_files = glob.glob(file_pattern)

if not all_files:
    print("no file 'test_results'.")
else:
    print(f"find {len(all_files)} merge...")
    combined_df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
    output_file = "merged_test_results.csv"
    combined_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"lưu tại: {output_file}")

no file 'test_results'.


In [3]:
df = pd.read_csv("merged_test_results.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10253 entries, 0 to 10252
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Test ID                10253 non-null  object 
 1   Question ID            9799 non-null   float64
 2   Question               9799 non-null   object 
 3   User Answer            9799 non-null   object 
 4   Correct Answer         9799 non-null   object 
 5   Is Correct             9799 non-null   object 
 6   IP Changed             10253 non-null  object 
 7   Copies Count           10253 non-null  int64  
 8   Pastes Count           10253 non-null  int64  
 9   Right Clicks Count     10253 non-null  int64  
 10  Window Switches Count  10253 non-null  int64  
 11  Test Duration          10253 non-null  float64
 12  Supicious              9759 non-null   object 
 13  Suspicious             454 non-null    float64
 14  ratio correct          454 non-null    float64
 15  Su

In [4]:
df = df.rename(columns={'Supicious': 'Suspicious'})
df.drop(columns=['Supicious.1'], inplace=True)

# Làm sạch dữ liệu #

#### Tổng hợp các câu trả lời đúng chia tổng số 20 câu hỏi để lấy tỉ lệ trả lời đúng, tỉ lệ được lưu trong cột 'ratio correct' ####

In [5]:
df['ratio correct'] = df.groupby('Test ID')['Is Correct'].transform(
    lambda x: (x == 'Yes').sum() / 20
)
df[['Test ID', 'Is Correct', 'ratio correct']].head(25)

,Test ID,Is Correct,ratio correct
0,01403B08,Yes,0.95
1,01403B08,Yes,0.95
2,01403B08,Yes,0.95
3,01403B08,Yes,0.95
4,01403B08,Yes,0.95
5,01403B08,Yes,0.95
6,01403B08,Yes,0.95
7,01403B08,Yes,0.95
8,01403B08,Yes,0.95
9,01403B08,Yes,0.95


#### Loại bỏ các cột không cần thiết ####

In [6]:
df = df.drop(columns=['Question ID', 'Question','User Answer','Correct Answer','Is Correct'])

#### Với mỗi test ID chỉ là những bản ghi lặp lại thế nên sẽ loại bỏ những bản ghi trùng lặp ####

#### Giờ với mỗi test ID sẽ là những bản ghi duy nhất chứa số điểm, thời gian, ... ####

In [7]:
df = df.drop_duplicates(subset=['Test ID'], keep='first')

In [8]:
df.head()

,Test ID,IP Changed,Copies Count,Pastes Count,Right Clicks Count,Window Switches Count,Test Duration,Suspicious,Suspicious,ratio correct
0,01403B08,No,0,0,0,0,11.67,no,NaN,0.95
20,04BECD6A,No,7,4,0,24,8.40,yes,NaN,0.95
40,04Y82P54,Yes,20,6,0,40,5.98,yes,NaN,0.90
60,0964B72D,No,0,0,0,5,10.45,no,NaN,0.80
80,0B640F58,No,0,0,0,0,29.05,no,NaN,0.95


#### Với những bài test có thời gian > 20 phút biến đổi nó trở về 20 ####

In [9]:
df['Test Duration'] = df['Test Duration'].clip(upper=20)
df.head(5)

,Test ID,IP Changed,Copies Count,Pastes Count,Right Clicks Count,Window Switches Count,Test Duration,Suspicious,Suspicious,ratio correct
0,01403B08,No,0,0,0,0,11.67,no,NaN,0.95
20,04BECD6A,No,7,4,0,24,8.40,yes,NaN,0.95
40,04Y82P54,Yes,20,6,0,40,5.98,yes,NaN,0.90
60,0964B72D,No,0,0,0,5,10.45,no,NaN,0.80
80,0B640F58,No,0,0,0,0,20.00,no,NaN,0.95


# Chuẩn hóa dữ liệu #

#### với 2 cột IP changed và Suspicious chuyển từ yes/no thành 1 và 0 ####

In [10]:
df['IP Changed'] = df['IP Changed'].astype(str).str.lower().map({'yes': 1, 'no': 0})
df['Suspicious'] = df['Suspicious'].astype(str).str.lower().map({'yes': 1, 'no': 0})

AttributeError: 'DataFrame' object has no attribute 'str'

In [ ]:
# Check for null values before dropping
print("Null values before dropping:")
print(df.isnull().sum())
print(f"\nTotal rows before: {len(df)}")

# Drop rows that contain any null values
df = df.dropna()

print(f"Total rows after: {len(df)}")
print("\nNull values after dropping:")
print(df.isnull().sum())

# Làm sạch dữ liệu #

#### Xóa những dòng có giá trị thiếu ####

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)
df.isnull().sum()

In [ ]:
df.info()

#### Biến đổi kiểu của Suspicious từ float sang int ####

In [ ]:
df['Suspicious'] = df['Suspicious'].astype('Int64')
df.info()

#### Lưu dataframe vào test_results_cleaned ####

In [ ]:
df.to_csv('test_results_cleaned.csv', index=False)

# Visualize #

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv('test_results_cleaned.csv')

In [ ]:
plt.hist(df['ratio correct'], bins=20)
plt.title('Score Distribution')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df['Suspicious'].value_counts().plot(kind='bar')
plt.title('Suspicious Distribution')
plt.show()

In [ ]:
# Ví dụ: trung bình score theo label
grouped = df.groupby('Suspicious')['ratio correct'].mean()
print(grouped)

grouped.plot(kind='bar')
plt.title('Average Score by Label')
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 8))
# Chỉ cần 1 dòng này thay thế cho plt.imshow và 2 vòng lặp for ở trên
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)

plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(df['Test Duration'], df['ratio correct'])
plt.xlabel('Time Spent')
plt.ylabel('Score')
plt.title('Time vs Score')
plt.show()

In [ ]:
df.boxplot(column='ratio correct', by='Suspicious')
plt.title('Score Distribution by Label')
plt.suptitle('')
plt.show()

# Tách dữ liệu thành tập train /test # 

- #### Với dữ liệu nhỏ cách phương pháp 80/20 hoặc 70/30 thông thường sẽ không hiệu quả ####
- #### Giải pháp tối ưu nhất sẽ là Stratified K-Fold Cross Validation với k = 5 ####

In [ ]:
from sklearn.model_selection import StratifiedKFold
df = pd.read_csv('test_results_cleaned.csv')
X = df.drop(columns=['Suspicious'])
y = df['Suspicious']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)